In [ ]:
import pandas as pd
import numpy as np


In [ ]:
df = pd.read_csv('/content/online_retail_Cleaned.csv', engine='python', on_bad_lines='skip')

In [ ]:
df.head(2)

In [ ]:
df.shape

In [ ]:
df.columns

In [ ]:
df.isnull().sum()

In [ ]:
df.info()

In [ ]:
# Check missing customer IDs
df['CustomerID'].isna().sum()

In [ ]:
# Convert InvoiceDate to datetime
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

In [ ]:
# Convert Price to numeric
df['Price'] = pd.to_numeric(df['Price'], errors='coerce')

In [ ]:
# Calculate sales for each row
df['Sales'] = df['Quantity'] * df['Price']

In [ ]:
df['CustomerID'] = pd.to_numeric(df['CustomerID'], errors='coerce')

In [ ]:
df['Quantity'] = pd.to_numeric(df['Quantity'], errors='coerce')

In [ ]:
# Filter and calculate recency, frequency, monetary

In [ ]:
# Filter out missing IDs and dates
customer_df = df.dropna(subset=['CustomerID', 'InvoiceDate']).copy()

# Calculate RFM Metrics
snapshot_date = customer_df['InvoiceDate'].max() + pd.Timedelta(days=1)
last_purchage = customer_df.groupby('CustomerID')['InvoiceDate'].max()

recency = (snapshot_date - last_purchage).dt.days
frequency = customer_df.groupby('CustomerID')['Invoice'].nunique()
monetary = customer_df.groupby('CustomerID')['Sales'].sum()

rfm = pd.DataFrame({
    'Recency': recency,
    'Frequency': frequency,
    'Monetary': monetary
}).reset_index()

rfm.head(10)

In [ ]:
def recency_score(days):
    if days <= 30:
        return 5
    elif days <= 60:
        return 4
    elif days <= 90:
        return 3
    elif days <= 180:
        return 2
    else:
        return 1

rfm['R_Score'] = rfm['Recency'].apply(recency_score)

In [ ]:
def frequency_score(orders):
    if orders >= 10:
        return 5
    elif orders >= 7:
        return 4
    elif orders >= 5:
        return 3
    elif orders >= 2:
        return 2
    else:
        return 1

rfm['F_Score'] = rfm['Frequency'].apply(frequency_score)

In [ ]:
def monetary_score(amount):
    if amount >= 5000:
        return 5
    elif amount >= 2000:
        return 4
    elif amount >= 1000:
        return 3
    elif amount >= 500:
        return 2
    else:
        return 1

rfm['M_Score'] = rfm['Monetary'].apply(monetary_score)

In [ ]:
rfm[['CustomerID', 'R_Score', 'F_Score', 'M_Score']].head(10)

In [ ]:
def customer_segment(row):

    if row['R_Score'] >= 4 and row['F_Score'] >= 4 and row['M_Score'] >= 4:
        return 'Champions'

    elif row['F_Score'] >= 4 and row['M_Score'] >= 4:
        return 'Loyal'

    elif row['R_Score'] <= 2:
        return 'At Risk'

    else:
        return 'Regular'


rfm['Segment'] = rfm.apply(customer_segment, axis=1)
rfm['IsChurned'] = rfm['Recency'] > 90

rfm[['CustomerID', 'R_Score', 'F_Score', 'M_Score', 'Segment', 'IsChurned']].head(10)

In [ ]:
print("Champions:", (rfm['Segment'] == 'Champions').sum())
print("Loyal:", (rfm['Segment'] == 'Loyal').sum())
print("At Risk:", (rfm['Segment'] == 'At Risk').sum())
print("Regular:", (rfm['Segment'] == 'Regular').sum())

In [ ]:
rfm.head()

In [ ]:
print("Total RFM customers:", len(rfm))

In [ ]:
print(rfm['Segment'].value_counts())

In [30]:

# 1. Unnecessary rows remove karo
df = df[(df['CustomerID'].notna()) & (df['Quantity'] > 0)].copy()

# 2. InvoiceDate ko date format mein convert karo
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

# 3. Revenue calculate karo
df['Revenue'] = df['Quantity'] * df['Price']

# 4. RFM columns customer data se merge karo
df_final = df.merge(
    rfm[['CustomerID', 'Recency', 'Frequency', 'Monetary',
         'R_Score', 'F_Score', 'M_Score', 'Segment', 'IsChurned']],
    on='CustomerID',
    how='left'
)


